# FastAPI — Update & Delete Endpoints

---

## 1) Updating a Patient — `PUT /edit/{patient_id}`

### What this endpoint does
- Accepts a `patient_id` as a **path parameter** (to identify which patient to update)
- Accepts a **request body** with only the fields the user wants to change
- Updates only those fields, leaving everything else untouched

---

### The Problem with the Existing Pydantic Model

Our original `Patient` model uses `...` (the ellipsis) in every `Field`, which means **all fields are required**:

```python
class Patient(BaseModel):
    id:     Annotated[str,   Field(...)]   # required
    name:   Annotated[str,   Field(...)]   # required
    age:    Annotated[int,   Field(...)]   # required
    # ... all fields required
```

If we reuse this model for updates, the client would have to send **every single field** even if they only want to change the city. That's bad UX.

---

### The Solution — A Separate `PatientUpdate` Model

Create a second model where **every field is `Optional`** with a `default=None`. This way the client only sends the fields they want to update:

```python
class PatientUpdate(BaseModel):
    # id is NOT included — it comes from the path parameter, not the request body
    name:   Annotated[Optional[str],                              Field(default=None)]
    gender: Annotated[Optional[Literal['male','female','others']], Field(default=None)]
    age:    Annotated[Optional[int],                              Field(default=None, gt=0)]
    city:   Annotated[Optional[str],                              Field(default=None)]
    height: Annotated[Optional[float],                            Field(default=None, gt=0)]
    weight: Annotated[Optional[float],                            Field(default=None, gt=0)]
```

> **Why no `id` field?** The `id` comes from the URL path (`/edit/{patient_id}`), so there's no need to send it in the request body again.

---

### Why `exclude_unset=True` in `model_dump()`?

When a client sends only `{ "city": "Mumbai" }`, Pydantic fills all other `Optional` fields with `None` (their default). If we call plain `model_dump()`, we get:

```python
{'name': None, 'gender': None, 'age': None, 'city': 'Mumbai', 'height': None, 'weight': None}
```

This would **overwrite** existing values with `None` — not what we want!

Using `model_dump(exclude_unset=True)` returns **only the fields the client actually sent**:

```python
{'city': 'Mumbai'}   # ✅ only what was sent
```

---

### Handling Computed Fields (`bmi` & `verdict`) After Update

After merging the updated fields into the existing data, the `bmi` and `verdict` values are now stale. We need to **recalculate them**.

The clean way to do this is:

```
existing data (dict)  →  Patient Pydantic object  →  bmi & verdict recomputed automatically  →  model_dump()  →  save back to file
```

We temporarily add `id` back into the dict before creating the `Patient` object (since the `Patient` model requires it), then exclude it again when saving.

---

### Full Update Endpoint

```python
@app.put('/edit/{patient_id}')
def update_patient(patient_id: str, patient_update: PatientUpdate):

    # Step 1 — Load existing data
    data = load_data()

    # Step 2 — Check if patient exists
    if patient_id not in data:
        raise HTTPException(status_code=404, detail='Patient not found')

    # Step 3 — Get current patient record
    existing_patient_info = data[patient_id]

    # Step 4 — Extract only the fields sent by the client (ignore unset None values)
    updated_fields = patient_update.model_dump(exclude_unset=True)

    # Step 5 — Merge updated fields into existing record
    for key, value in updated_fields.items():
        existing_patient_info[key] = value
        # finds the matching key in the old data and replaces its value

    # Step 6 — Recompute bmi & verdict by creating a fresh Patient Pydantic object
    # We need to add 'id' temporarily because the Patient model requires it
    existing_patient_info['id'] = patient_id
    patient_pydantic_obj = Patient(**existing_patient_info)

    # Step 7 — Convert back to dict (bmi & verdict are now recalculated)
    updated_data = patient_pydantic_obj.model_dump(exclude={'id'})

    # Step 8 — Save back to file
    data[patient_id] = updated_data
    save_data(data)

    return JSONResponse(status_code=200, content={'message': 'Patient updated successfully'})
```

---

### Update Flow — Visual Summary

```
Client sends partial JSON  →  PatientUpdate model (only sent fields)
                                        ↓
                           model_dump(exclude_unset=True)
                                        ↓
                           Merge into existing patient dict
                                        ↓
                           Create Patient object  ←  bmi & verdict recomputed
                                        ↓
                           model_dump(exclude={'id'})
                                        ↓
                           Save updated dict to JSON file
```

---

## 2) Deleting a Patient — `DELETE /delete/{patient_id}`

### What this endpoint does
- Accepts a `patient_id` as a path parameter
- Checks if the patient exists
- Deletes the record and saves the file

```python
@app.delete('/delete/{patient_id}')
def delete_patient(patient_id: str):

    # Step 1 — Load existing data
    data = load_data()

    # Step 2 — Check if patient exists
    if patient_id not in data:
        raise HTTPException(status_code=404, detail='Patient not found')

    # Step 3 — Delete the patient record
    del data[patient_id]

    # Step 4 — Save updated data
    save_data(data)

    return JSONResponse(status_code=200, content={'message': 'Patient deleted successfully'})
```

---

## 3) Complete Code

```python
from fastapi import FastAPI, Path, HTTPException, Query
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, computed_field
from typing import Annotated, Literal, Optional
import json

# ── Pydantic Models ───────────────────────────────────────────────────────────

class Patient(BaseModel):
    id:     Annotated[str,   Field(..., description='Unique ID of the patient', example='P001')]
    name:   Annotated[str,   Field(..., description='Full name of the patient')]
    gender: Annotated[Literal['male', 'female', 'others'], Field(..., description='Gender of the patient')]
    age:    Annotated[int,   Field(..., description='Age of the patient', gt=0, lt=101)]
    city:   Annotated[str,   Field(..., description='City of the patient')]
    height: Annotated[float, Field(..., description='Height of the patient in metres')]
    weight: Annotated[float, Field(..., description='Weight of the patient in kg')]

    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / (self.height ** 2), 2)

    @computed_field
    @property
    def verdict(self) -> str:
        if self.bmi < 18.5:
            return 'Underweight'
        elif self.bmi < 25:
            return 'Normal'
        else:
            return 'Obese'


class PatientUpdate(BaseModel):
    name:   Annotated[Optional[str],                               Field(default=None)]
    gender: Annotated[Optional[Literal['male', 'female', 'others']], Field(default=None)]
    age:    Annotated[Optional[int],                               Field(default=None, gt=0)]
    city:   Annotated[Optional[str],                               Field(default=None)]
    height: Annotated[Optional[float],                             Field(default=None, gt=0)]
    weight: Annotated[Optional[float],                             Field(default=None, gt=0)]


# ── App & Helpers ─────────────────────────────────────────────────────────────

app = FastAPI()

def load_data():
    with open('patient.json', 'r') as f:
        return json.load(f)

def save_data(data):
    with open('patient.json', 'w') as f:
        json.dump(data, f)


# ── Endpoints ─────────────────────────────────────────────────────────────────

@app.post('/create')
def create_patient(patient: Patient):
    data = load_data()
    if patient.id in data:
        raise HTTPException(status_code=400, detail='Patient already exists')
    data[patient.id] = patient.model_dump(exclude={'id'})
    save_data(data)
    return JSONResponse(status_code=201, content={'message': 'Patient created successfully'})


@app.put('/edit/{patient_id}')
def update_patient(patient_id: str, patient_update: PatientUpdate):
    data = load_data()
    if patient_id not in data:
        raise HTTPException(status_code=404, detail='Patient not found')
    existing_patient_info = data[patient_id]
    updated_fields = patient_update.model_dump(exclude_unset=True)
    for key, value in updated_fields.items():
        existing_patient_info[key] = value
    existing_patient_info['id'] = patient_id
    patient_pydantic_obj = Patient(**existing_patient_info)
    data[patient_id] = patient_pydantic_obj.model_dump(exclude={'id'})
    save_data(data)
    return JSONResponse(status_code=200, content={'message': 'Patient updated successfully'})


@app.delete('/delete/{patient_id}')
def delete_patient(patient_id: str):
    data = load_data()
    if patient_id not in data:
        raise HTTPException(status_code=404, detail='Patient not found')
    del data[patient_id]
    save_data(data)
    return JSONResponse(status_code=200, content={'message': 'Patient deleted successfully'})
```

---

## Quick Reference — HTTP Methods

| Method | Endpoint | Purpose | Request Body |
|--------|----------|---------|--------------|
| `POST` | `/create` | Add a new patient | Full `Patient` model |
| `PUT` | `/edit/{patient_id}` | Update an existing patient | Partial `PatientUpdate` model |
| `DELETE` | `/delete/{patient_id}` | Remove a patient | None |

---

## Key Concepts Recap

| Concept | Why it's used |
|---------|--------------|
| `PatientUpdate` with `Optional` fields | Allows partial updates — client sends only what they want to change |
| `model_dump(exclude_unset=True)` | Returns only fields the client actually sent, ignoring `None` defaults |
| Re-creating `Patient` object after merge | Forces recomputation of `bmi` and `verdict` computed fields |
| `del data[patient_id]` | Native Python dict deletion — simple and clean |